# Batch Evaluation
> Evaluate multiple samples at once and aggregate the results.

`BatchEvaluator` iterates over an `EvaluationDataset` and
produces aggregate metrics across all samples, making it easy
to benchmark your pipeline on a test set.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.evaluation import (
    BatchEvaluator,
    PipelineEvaluator,
    RetrievalMetricsCalculator,
    LLMRAGEvaluator,
)
from anchor.evaluation.models import (
    EvaluationSample,
    EvaluationDataset,
    RAGMetrics,
)
from anchor.models import ContextItem, SourceType

## Build the Pipeline Evaluator
Reuse the same components from the previous notebook.

In [ ]:
calculator = RetrievalMetricsCalculator()

def mock_judge(query, answer, contexts, ground_truth=None):
    return RAGMetrics(
        faithfulness=0.9,
        relevancy=0.85,
        context_precision=0.8,
        context_recall=0.75,
    )

rag_evaluator = LLMRAGEvaluator(eval_fn=mock_judge)
pipeline_eval = PipelineEvaluator(
    retrieval_calculator=calculator,
    rag_evaluator=rag_evaluator,
)

print(f"Pipeline evaluator ready: {type(pipeline_eval).__name__}")

## Create Evaluation Samples
Each `EvaluationSample` bundles a query, answer, contexts,
retrieved items, relevant IDs, and ground truth.

In [ ]:
def make_retrieved(n=5):
    return [
        ContextItem(
            id=f"doc-{i}",
            content=f"Content {i}",
            source=SourceType.RETRIEVAL,
            score=1.0 - i * 0.1,
            priority=5,
            token_count=10,
        )
        for i in range(n)
    ]

samples = [
    EvaluationSample(
        query="What is context engineering?",
        answer="Context engineering designs LLM inputs.",
        contexts=["Context engineering is about input design."],
        retrieved=make_retrieved(),
        relevant=["doc-0", "doc-1"],
        ground_truth="Context engineering is the systematic design of LLM inputs.",
    ),
    EvaluationSample(
        query="What is RAG?",
        answer="RAG combines retrieval with generation.",
        contexts=["RAG stands for Retrieval-Augmented Generation."],
        retrieved=make_retrieved(),
        relevant=["doc-0", "doc-2", "doc-3"],
        ground_truth="RAG augments LLM generation with retrieved context.",
    ),
    EvaluationSample(
        query="What are embeddings?",
        answer="Embeddings are vector representations of text.",
        contexts=["Embeddings map text to dense vectors."],
        retrieved=make_retrieved(),
        relevant=["doc-0"],
        ground_truth="Embeddings encode text as numerical vectors.",
    ),
]

print(f"Created {len(samples)} evaluation samples.")
for i, s in enumerate(samples):
    print(f"  Sample {i}: query={s.query!r:.40}")

## Build an Evaluation Dataset
`EvaluationDataset` is a lightweight container with an
`.add_sample()` method.

In [ ]:
dataset = EvaluationDataset()
for s in samples:
    dataset.add_sample(s)

print(f"Dataset size: {len(dataset.samples)} samples")

## Run Batch Evaluation
`BatchEvaluator.evaluate_batch()` processes every sample and
returns aggregated metrics.

In [ ]:
batch = BatchEvaluator(evaluator=pipeline_eval)
aggregated = batch.evaluate_batch(samples=dataset.samples)

print("Aggregated retrieval metrics:")
print(f"  Avg Precision: {aggregated.avg_precision:.3f}")
print(f"  Avg Recall:    {aggregated.avg_recall:.3f}")
print(f"  Avg MRR:       {aggregated.avg_mrr:.3f}")
print(f"  Avg nDCG:      {aggregated.avg_ndcg:.3f}")

## Inspect Individual Results
The aggregated object may expose per-sample results for
deeper analysis.

In [ ]:
print(f"Batch evaluator type: {type(batch).__name__}")
print(f"Aggregated type:      {type(aggregated).__name__}")
print(f"Samples processed:    {len(dataset.samples)}")

## Key Takeaways

- `EvaluationSample` bundles all inputs needed for one evaluation.
- `EvaluationDataset` collects samples into a reusable test set.
- `BatchEvaluator` runs the pipeline evaluator over every sample
  and returns aggregate precision, recall, MRR, and nDCG.
- Use batch evaluation to benchmark pipeline changes on a fixed
  dataset.